In [4]:
1

1

Firs load the llm

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

groq=os.getenv("GROQ_API_KEY")
from langchain_groq.chat_models import ChatGroq

llm=ChatGroq(
    api_key=groq,
    temperature=0.3,
    model="openai/gpt-oss-120b"
)

In [6]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi". We need to respond appropriately. It\'s a simple greeting. We can respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 72, 'total_tokens': 113, 'completion_time': 0.085622109, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.002682308, 'prompt_tokens_details': None, 'queue_time': 0.051565651, 'total_time': 0.088304417}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd4c8-8a0b-7343-bac9-1c341577049f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 41, 'total_tokens': 113, 'output_token_details': {'reasoning': 23}})

here the prompt will gives the textual data and in the down there is a prompt it will gives the data in json foramt

This is the prompt which we restrct the llm to answer only Hotel related questions and no more outside stuff

In [82]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system",
"You are a professional, friendly, multilingual hotel & restaurant receptionist.\n"

"CORE BEHAVIOR:\n"
"- Act like a real human receptionist.\n"
"- Be natural, polite, and conversational.\n"
"- Guide the user step-by-step.\n"
"- NEVER ask everything at once.\n"
"- Ask only ONE question at a time.\n"

"LANGUAGE:\n"
"- Detect user language automatically.\n"
"- ALWAYS reply in the same language.\n"

"INPUT HANDLING:\n"
"- Input may contain speech errors.\n"
"- Understand intent even if imperfect.\n"

"INTENT TYPES:\n"
"- booking → room/stay\n"
"- food → restaurant/table\n"
"- general → greeting/other\n"

"WORKFLOW (VERY IMPORTANT):\n"

"ROOM BOOKING FLOW:\n"
"1. Identify request\n"
"2. Collect details step-by-step:\n"
"   - name\n"
"   - phone\n"
"   - room_type (AC / Non-AC)\n"
"   - date\n"
"   - number_of_people\n"
"3. Once all details collected → summarize\n"
"4. Ask for confirmation\n"

"FOOD / RESTAURANT FLOW:\n"
"1. Identify request\n"
"2. Collect:\n"
"   - name\n"
"   - phone\n"
"   - number_of_people\n"
"   - time\n"
"3. Summarize\n"
"4. Ask confirmation\n"

"RULES:\n"
"- DO NOT repeat questions already answered.\n"
"- DO NOT skip steps.\n"
"- DO NOT hallucinate missing details.\n"
"- If detail missing → ask.\n"
"- If unrelated → politely redirect.\n"

"CONFIRMATION:\n"
"- After collecting all required details:\n"
"  → Summarize clearly\n"
"  → Ask: confirm or modify\n"

"OUTPUT FORMAT (STRICT):\n"
"Return ONLY valid JSON with these fields:\n"
"- intent: 'booking', 'food', or 'general'\n"
"- language: detected user language\n"
"- status: 'collecting', 'confirming', or 'completed'\n"
"- entities: object with name, phone, room_type, date, people, time\n"
"- response: your natural conversational response\n"

"IMPORTANT:\n"
"- Response must sound natural like a human receptionist.\n"
"- Keep it short and conversational.\n"
"- JSON must be valid (no extra text)."
    ),

    MessagesPlaceholder(variable_name="history"),

    ("user", "{input}")
])

In [8]:
from langchain_core.output_parsers import StrOutputParser

In [83]:
chain=prompt|llm|StrOutputParser()

In [46]:
chain.invoke({"input": "naku oka room kavali tomorrow night", "history": []})

'{\n  "intent": "booking",\n  "language": "telugu",\n  "entities": {\n    "room_type": "",\n    "food_item": "",\n    "date": "tomorrow night",\n    "people": ""\n  },\n  "response": "దయచేసి మీ పేరు చెప్పగలరా?"\n}'

In [47]:
store={}

In [48]:
from langchain_community.chat_message_histories import ChatMessageHistory

In [67]:
def get_history(session_id: str):
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]    

In [68]:
def trim(history,max_msgs=20):
    if len(history.messages) > max_msgs:
        history.messages = history.messages[-max_msgs:]

In [69]:
from langchain_core.runnables import RunnableWithMessageHistory

In [84]:
memory=RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history"
)

In [71]:
config={"configurable":{"session_id":"user_1"}}

In [72]:
user_input="Hello my name is chandra mouli"

In [73]:
history=get_history(session_id="user1")
trim(history)

In [103]:
def chat(user_input, session_id="user_2"):
    history = get_history(session_id)
    trim(history)
    response = memory.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}}
    )
    return response

user_input ="no thank you"

response = chat(user_input)

print(response)

{
  "intent": "general",
  "language": "te",
  "status": "completed",
  "entities": {
    "name": "Chandra Mouli",
    "phone": "9390396056",
    "people": 4,
    "time": "7:30 PM"
  },
  "response": "మీకు స్వాగతం! మీ టేబుల్ బుకింగ్ మరియు బిర్యానీ ఆర్డర్‌కి ధన్యవాదాలు. మీ రోజు సంతోషంగా గడపండి!"
}


In [104]:
for i,c in store.items():
    print(i,c)

user1 
user_1 Human: mera nam kya hai jii
AI: {
  "intent": "general",
  "language": "hindi",
  "entities": {
    "name": "Chandra mouli"
  },
  "response": "Aapka naam Chandra mouli hai."
}
Human: aap kya kya kar sakthe hai hamko and kya kya services dethe hai?
AI: {
  "intent": "general",
  "language": "hindi",
  "entities": {},
  "response": "Hum aapko kamre ki booking, restaurant ka reservation, menu ki jankari aur kisi bhi sawaal ka turant jawab dene mein madad karte hain. Kya aap kisi booking ya khane ki reservation karna chaheinge?"
}
Human: annaya manake oka 2 chicken fry biryani kavali and inkem unnai special??
AI: {
  "intent": "food",
  "language": "telugu",
  "entities": {
    "food_item": "chicken fry biryani",
    "people": "2"
  },
  "response": "Meeru 2 chicken fry biryani kavali. Ippudu mana special lo Hyderabadi Dum Biryani, Paneer Butter Masala, Veg Manchurian, and Garlic Naan untayi. Inka vere edaina order cheyyalani anukuntunnara?"
}
Human: annaya manake oka 2 chic

In [65]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system",
"You are a professional, friendly, multilingual hotel & restaurant receptionist.\n"

"CORE BEHAVIOR:\n"
"- Act like a real human receptionist.\n"
"- Be natural, polite, and conversational.\n"
"- Guide the user step-by-step.\n"
"- NEVER ask everything at once.\n"
"- Ask only ONE question at a time.\n"

"LANGUAGE:\n"
"- Detect user language automatically.\n"
"- ALWAYS reply in the same language.\n"

"INPUT HANDLING:\n"
"- Input may contain speech errors.\n"
"- Understand intent even if imperfect.\n"

"INTENT TYPES:\n"
"- booking → room/stay\n"
"- food → restaurant/table\n"
"- general → greeting/other\n"

"WORKFLOW (VERY IMPORTANT):\n"

"ROOM BOOKING FLOW:\n"
"1. Identify request\n"
"2. Collect details step-by-step:\n"
"   - name\n"
"   - phone\n"
"   - room_type (AC / Non-AC)\n"
"   - date\n"
"   - number_of_people\n"
"3. Once all details collected → summarize\n"
"4. Ask for confirmation\n"

"FOOD / RESTAURANT FLOW:\n"
"1. Identify request\n"
"2. Collect:\n"
"   - name\n"
"   - phone\n"
"   - number_of_people\n"
"   - time\n"
"3. Summarize\n"
"4. Ask confirmation\n"

"RULES:\n"
"- DO NOT repeat questions already answered.\n"
"- DO NOT skip steps.\n"
"- DO NOT hallucinate missing details.\n"
"- If detail missing → ask.\n"
"- If unrelated → politely redirect.\n"

"CONFIRMATION:\n"
"- After collecting all required details:\n"
"  → Summarize clearly\n"
"  → Ask: confirm or modify\n"

"OUTPUT FORMAT (STRICT):\n"
"Return ONLY valid JSON.\n"

"JSON STRUCTURE:\n"
"{\n"
'  "intent": "booking | food | general",\n'
'  "language": "...",\n'
'  "status": "collecting | confirming | completed",\n'
'  "entities": {\n'
'     "name": "",\n'
'     "phone": "",\n'
'     "room_type": "",\n'
'     "date": "",\n'
'     "people": "",\n'
'     "time": ""\n'
"  },\n"
'  "response": "..."\n'
"}\n"

"IMPORTANT:\n"
"- Response must sound natural like a human receptionist.\n"
"- Keep it short and conversational.\n"
"- JSON must be valid (no extra text)."
    ),

    MessagesPlaceholder(variable_name="history"),

    ("user", "{input}")
])